# Fraud Detection — Data Preparation

This notebook explores the fraud modeling dataset and prepares the training and validation datasets used for fraud detection modeling.

The main goals are to:
- inspect the available fraud features,
- understand the time coverage and fraud distribution,
- define the model features and target,
- create a time-based training and validation split.

In [1]:
import pandas as pd

from finsight.database import connect_to_database
from finsight.fraud_data import (
    download_train_data,
    download_validation_data,
)

## Database Connection

In [2]:
engine = connect_to_database()

Connected!


## Fraud Distribution Over Time

In [3]:
dataset_query = """
select
    *
from analytics.fraud_ml_features
limit 5
"""

dataset = pd.read_sql(dataset_query, engine)

dataset

,transaction_key,transaction_timestamp,amount,mcc_key,use_chip,merchant_id,transaction_hour,day_of_week,is_weekend,user_previous_transaction_count,user_previous_avg_amount,amount_vs_user_avg,card_previous_transaction_count,card_previous_avg_amount,amount_vs_card_avg,is_fraud
0,9564726,2010-01-01 13:25:00,10.24,4,Swipe Transaction,21491,13.0,5.0,0,0,NaN,NaN,0,NaN,NaN,False
1,8733454,2010-01-02 06:26:00,138.32,16,Swipe Transaction,60569,6.0,6.0,1,2,53.88,84.44,2,53.88,84.44,False
2,3536380,2010-01-02 07:28:00,181.83,53,Swipe Transaction,98050,7.0,6.0,1,4,-40.98,222.81,4,-40.98,222.81,False
3,12683385,2010-01-02 13:10:00,154.84,81,Swipe Transaction,71883,13.0,6.0,1,5,3.58,151.26,5,3.58,151.26,False
4,5201105,2010-01-02 13:24:00,-191.00,81,Swipe Transaction,71883,13.0,6.0,1,6,28.79,-219.79,6,28.79,-219.79,False


In [4]:
dataset_info_query = """
select
    min(transaction_timestamp) as min_date,
    max(transaction_timestamp) as max_date,
    count(*) as number_of_transactions
from analytics.fraud_ml_features
"""

dataset_info = pd.read_sql(dataset_info_query, engine)

dataset_info

,min_date,max_date,number_of_transactions
0,2010-01-01 00:01:00,2019-10-31 23:57:00,8914963


In [5]:
fraud_by_year_query = """
select
    extract(year from transaction_timestamp)::int as year,
    count(*) as transactions,
    count(*) filter (where is_fraud = true) as fraud_transactions,
    round(count(*) filter (where is_fraud = true)::numeric / count(*) * 100, 4) as fraud_rate
from analytics.fraud_ml_features
group by extract(year from transaction_timestamp)
order by year
"""

fraud_by_year = pd.read_sql(fraud_by_year_query, engine)

fraud_by_year

,year,transactions,fraud_transactions,fraud_rate
0,2010,831529,2573,0.3094
1,2011,863428,37,0.0043
2,2012,885421,923,0.1042
3,2013,907304,1337,0.1474
4,2014,915073,664,0.0726
5,2015,930224,2189,0.2353
6,2016,932762,2448,0.2624
7,2017,937284,172,0.0184
8,2018,934599,1629,0.1743
9,2019,777339,1360,0.1750


In [6]:
fraud_by_month_query = """
select
    date_trunc('month', transaction_timestamp) as month,
    count(*) as transactions,
    count(*) filter (where is_fraud = true) as fraud_transactions,
    round(count(*) filter (where is_fraud = true)::numeric / count(*) * 100, 4) as fraud_rate
from analytics.fraud_ml_features
group by date_trunc('month', transaction_timestamp)
order by month
"""

fraud_by_month = pd.read_sql(fraud_by_month_query, engine)

fraud_by_month

,month,transactions,fraud_transactions,fraud_rate
0,2010-01-01,68044,107,0.1573
1,2010-02-01,62816,259,0.4123
2,2010-03-01,69202,261,0.3772
3,2010-04-01,66729,237,0.3552
4,2010-05-01,70210,274,0.3903
...,...,...,...,...
113,2019-06-01,77089,132,0.1712
114,2019-07-01,79891,104,0.1302
115,2019-08-01,79513,165,0.2075
116,2019-09-01,76977,90,0.1169


In [7]:
fraud_by_month["fraud_transactions"].describe()

count    118.000000
mean     112.983051
std       99.149717
min        0.000000
25%        0.000000
50%      116.000000
75%      192.750000
max      423.000000
Name: fraud_transactions, dtype: float64

In [8]:
print("Months with 0 fraud:",
      (fraud_by_month["fraud_transactions"] == 0).sum())

print("Months with < 10 fraud:",
      (fraud_by_month["fraud_transactions"] < 10).sum())

print("Min fraud rate:",
      fraud_by_month["fraud_rate"].min())

print("Max fraud rate:",
      fraud_by_month["fraud_rate"].max())

Months with 0 fraud: 39
Months with < 10 fraud: 40
Min fraud rate: 0.0
Max fraud rate: 0.5278


In [9]:
zero_fraud_months = fraud_by_month[
    fraud_by_month["fraud_transactions"] == 0
]

zero_fraud_months

,month,transactions,fraud_transactions,fraud_rate
13,2011-02-01,65223,0,0.0
14,2011-03-01,72224,0,0.0
15,2011-04-01,70777,0,0.0
16,2011-05-01,72667,0,0.0
17,2011-06-01,71162,0,0.0
18,2011-07-01,73440,0,0.0
19,2011-08-01,74420,0,0.0
20,2011-09-01,71418,0,0.0
21,2011-10-01,73877,0,0.0
22,2011-11-01,71564,0,0.0


## Feature Selection

Before training a model, the input features (`X`) must be separated from the target (`y`).

Three columns are excluded from the model features:

- `transaction_key` — a technical transaction identifier rather than a behavioral feature,
- `transaction_timestamp` — used to create the time-based data split; temporal information is already represented by derived features such as `transaction_hour`, `day_of_week` and `is_weekend`,
- `is_fraud` — the target variable that the model will learn to predict.

The remaining columns are used as model features.

In [10]:
dataset.columns.tolist()

['transaction_key',
 'transaction_timestamp',
 'amount',
 'mcc_key',
 'use_chip',
 'merchant_id',
 'transaction_hour',
 'day_of_week',
 'is_weekend',
 'user_previous_transaction_count',
 'user_previous_avg_amount',
 'amount_vs_user_avg',
 'card_previous_transaction_count',
 'card_previous_avg_amount',
 'amount_vs_card_avg',
 'is_fraud']

## Training Set

The training dataset contains transactions before 2018.

Because fraud transactions are rare, all available fraud observations from the training period are retained, while a deterministic sample of non-fraud transactions is used for training.

The reusable data-loading logic is implemented in `finsight.fraud_data`.

In [11]:
X_train, y_train = download_train_data(engine)

In [12]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTraining proportions:")
print(y_train.value_counts(normalize=True))

X_train: (310343, 13)
y_train: (310343,)

Training target:
is_fraud
False    300000
True      10343
Name: count, dtype: int64

Training proportions:
is_fraud
False    0.966672
True     0.033328
Name: proportion, dtype: float64


## Validation Set

The validation dataset contains transactions from 2018.

Unlike the training dataset, the validation set keeps the observed class distribution and is not undersampled. This allows the model to be evaluated on data that better reflects the original fraud frequency during the validation period.

In [13]:
X_val, y_val = download_validation_data(engine)

In [14]:
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nValidation target:")
print(y_val.value_counts())

print("\nValidation proportions:")
print(y_val.value_counts(normalize=True))

X_val: (934599, 13)
y_val: (934599,)

Validation target:
is_fraud
False    932970
True       1629
Name: count, dtype: int64

Validation proportions:
is_fraud
False    0.998257
True     0.001743
Name: proportion, dtype: float64
